In [ ]:
%%writefile object_detection_app.py
import streamlit as st
from ultralytics import YOLO
import cv2
import requests
import numpy as np
from PIL import Image
from io import BytesIO

# Set up page layout
st.set_page_config(page_title="Object Detection App", page_icon="🔍", layout="centered")

st.title("🔍 AI Object Detection App")
st.caption("Enter an image URL to detect objects instantly using a pre-trained YOLOv8 PyTorch network.")

@st.cache_resource
def load_yolo_model():
    return YOLO("yolov8n.pt")

model = load_yolo_model()

# Stable text input box instead of FileUploader component
url_input = st.text_input(
    "Paste Image URL here:",
    value="https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg"
)

if url_input:
    try:
        # Fetch the image safely from the web
        response = requests.get(url_input, timeout=10)
        image = Image.open(BytesIO(response.content))
        st.image(image, caption="Target Image Source", use_container_width=True)

        if st.button("Run Object Detection"):
            with st.spinner("Analyzing image..."):
                img_array = np.array(image)

                # Run inference with default safe threshold (0.45)
                results = model.predict(source=img_array, conf=0.45)

                # Draw bounding boxes with labels and confidence scores
                annotated_img_bgr = results[0].plot()
                annotated_img_rgb = cv2.cvtColor(annotated_img_bgr, cv2.COLOR_BGR2RGB)

                st.success("Detection Complete!")
                st.image(annotated_img_rgb, caption="Processed Image with Detections", use_container_width=True)

                st.subheader("Detected Objects Summary:")
                boxes = results[0].boxes
                if len(boxes) == 0:
                    st.write("No objects detected.")
                else:
                    for box in boxes:
                        class_id = int(box.cls[0])
                        class_name = model.names[class_id]
                        confidence_score = float(box.conf[0])
                        st.write(f"● **{class_name.capitalize()}** — Confidence: {confidence_score:.2%}")
    except Exception as e:
        st.error(f"Could not load image from URL. Please ensure it is a valid direct link to a JPG/PNG file. Error: {e}")

Overwriting object_detection_app.py


In [ ]:
# Print your public IP address for authentication
print("Your Public IP Address is:")
!curl ipv4.icanhazip.com

# Boot up the server instance cleanly
import os
os.system("nohup streamlit run object_detection_app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /dev/null 2>&1 &")

# Expose port 8501
!npx localtunnel --port 8501

Your Public IP Address is:
34.87.172.67
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://slimy-frogs-buy.loca.lt
^C
